# ScamShield AI
### Payment Scam Detection with Strands Decider 2B and Amazon Bedrock

**Agent Experience Hackathon — AWS User Group Madurai Strands Decider kit**

ScamShield AI takes any payment transaction description in plain language and classifies it as:
- **HIGH RISK** — scam indicators detected, do not proceed
- **NEEDS REVIEW** — some signals present, verify before acting
- **LOW RISK** — no major scam indicators found

Strands Decider 2B answers three calibrated risk questions *before* the LLM composes the reply,
running ~7x faster than asking the LLM the same questions.

**Decision pattern:** `#policy`  
**Tags:** `#strands-decider-2b` `#policy`

---

> **No API keys in this notebook.** Keys are read from `.env` in the project root (written by `./kit setup`).
> Run `./kit setup` first if you have not done so.

## 1. Setup — install dependencies and load environment

In [ ]:
# Install dependencies if running outside the kit's venv
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "strands-agents", "python-dotenv"], check=True)
print("Dependencies ready.")

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

# Load .env from the project root (one level up if running from a notebooks/ folder)
env_path = Path(".") / ".env"
if not env_path.exists():
    env_path = Path("..") / ".env"

if env_path.exists():
    load_dotenv(env_path)
    print(f"Loaded .env from {env_path.resolve()}")
    print(f"LLM provider : {os.environ.get('LLM_PROVIDER', 'bedrock')}")
    print(f"Decider mode : {os.environ.get('DECIDER_MODE', 'ugmdu')}")
    key_set = bool(os.environ.get('DECIDER_API_KEY'))
    print(f"API key set  : {key_set}")
    if not key_set:
        print("WARNING: DECIDER_API_KEY not set. Run ./kit setup to configure.")
else:
    print("WARNING: .env not found. Run ./kit setup to create it.")

## 2. The reusable decision pattern

Strands Decider 2B reads the transaction description and answers **three noul (yes/no) questions** in a single forward pass.
Each question returns a calibrated probability — not a freeform guess.

In [ ]:
# REUSABLE DECISION PATTERN: payment scam risk classifier  (#policy)
#
# Copy these QUESTIONS and ScamPolicyCheck into your own agent.
# Reword the option descriptions for your domain (e.g. phishing email, fake job offer).
# Adjust HIGH_RISK_THRESHOLD and REVIEW_THRESHOLD to tune sensitivity.

from decider_kit import Decider

PATTERN_TAG = "#policy"

HIGH_RISK_THRESHOLD = 0.60   # any signal above this -> High Risk
REVIEW_THRESHOLD    = 0.40   # any signal above this (but below HIGH) -> Needs Review

QUESTIONS = {
    "unknown_recipient": Decider.noul(
        "Is the recipient of this payment unknown, unverified, or not personally known to the sender?",
        {
            "true":  "the person or entity requesting payment is a stranger, has an unfamiliar name, "
                     "or cannot be verified through a known contact or official channel",
            "false": "the recipient is a known contact, a recognized business, or a verified official entity",
        },
    ),
    "urgency_pressure": Decider.noul(
        "Does the message use urgency, time pressure, or emotional pressure to push the sender to act immediately?",
        {
            "true":  "phrases like 'act now', 'urgent', 'last chance', 'emergency', 'or your account will be closed', "
                     "or emotional appeals that discourage taking time to verify",
            "false": "no time pressure; the sender is free to verify before acting",
        },
    ),
    "unusual_payment": Decider.noul(
        "Is the payment amount, method, or reason unusual or suspicious for a legitimate transaction?",
        {
            "true":  "very large or round amounts with no clear reason, requests for gift cards, "
                     "wire transfers to unknown accounts, requests to share OTPs or PINs, "
                     "or a reason that does not match the payment method",
            "false": "normal amount for the stated purpose, standard payment method, no request for sensitive info",
        },
    ),
}

print("Decision questions defined:")
for name in QUESTIONS:
    print(f"  - {name}")

In [ ]:
class ScamPolicyCheck:
    """Runs the three-question policy check and returns a structured verdict."""

    def __init__(self, decider: Decider):
        self.decider = decider
        self.log: list[dict] = []

    def check(self, transaction_description: str) -> dict:
        a = self.decider.ask(transaction_description, QUESTIONS)

        p_unknown = a["unknown_recipient"].p_yes or 0.0
        p_urgency = a["urgency_pressure"].p_yes  or 0.0
        p_unusual = a["unusual_payment"].p_yes   or 0.0
        max_signal = max(p_unknown, p_urgency, p_unusual)
        decider_ms = round(self.decider.last_latency_ms or 0)

        if   max_signal >= HIGH_RISK_THRESHOLD: verdict = "high_risk"
        elif max_signal >= REVIEW_THRESHOLD:    verdict = "needs_review"
        else:                                   verdict = "low_risk"

        flags = []
        if p_unknown >= REVIEW_THRESHOLD: flags.append("unknown_recipient")
        if p_urgency >= REVIEW_THRESHOLD: flags.append("urgency_pressure")
        if p_unusual >= REVIEW_THRESHOLD: flags.append("unusual_payment")

        record = {
            "verdict":           verdict,
            "flags":             flags,
            "unknown_recipient": round(p_unknown,  3),
            "urgency_pressure":  round(p_urgency,  3),
            "unusual_payment":   round(p_unusual,  3),
            "confidence":        round(max_signal, 3),
            "decider_ms":        decider_ms,
        }
        self.log.append(record)
        return record

print("ScamPolicyCheck class ready.")

## 3. Run the decider directly (no LLM)

This shows the raw calibrated probabilities Strands Decider 2B returns for each signal.
No LLM call needed — just one forward pass through the 2B model.

In [ ]:
import json

decider = Decider()   # reads DECIDER_MODE, DECIDER_URL, DECIDER_API_KEY from .env
policy  = ScamPolicyCheck(decider)

test_cases = [
    "I received a payment request from an unknown person asking me to transfer Rs.15,000 urgently.",
    "My friend Priya asked me to pay her Rs.500 for lunch via UPI, she is a saved contact.",
    "Someone claiming to be from the Income Tax department said I owe Rs.50,000 and must pay right now or face arrest.",
    "A work-from-home recruiter I found online is asking for a Rs.500 registration fee before my first task.",
    "Paying my electricity bill of Rs.1,340 through the official BESCOM portal.",
]

results = []
for tx in test_cases:
    r = policy.check(tx)
    results.append({"transaction": tx, **r})

print(json.dumps(results, indent=2))

## 4. Visualise the risk signals

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

labels   = [r["transaction"][:55] + "..." for r in results]
unknown  = [r["unknown_recipient"] for r in results]
urgency  = [r["urgency_pressure"]  for r in results]
unusual  = [r["unusual_payment"]   for r in results]
verdicts = [r["verdict"]           for r in results]

x     = np.arange(len(labels))
width = 0.25

verdict_colors = {"high_risk": "#d62728", "needs_review": "#ff7f0e", "low_risk": "#2ca02c"}
bar_colors = [verdict_colors[v] for v in verdicts]

fig, ax = plt.subplots(figsize=(13, 5))
b1 = ax.bar(x - width, unknown, width, label="Unknown recipient", color="#4c72b0", alpha=0.85)
b2 = ax.bar(x,         urgency, width, label="Urgency pressure",  color="#dd8452", alpha=0.85)
b3 = ax.bar(x + width, unusual, width, label="Unusual payment",   color="#55a868", alpha=0.85)

ax.axhline(HIGH_RISK_THRESHOLD, color="red",    linestyle="--", linewidth=1, label="High Risk threshold (0.60)")
ax.axhline(REVIEW_THRESHOLD,    color="orange", linestyle="--", linewidth=1, label="Review threshold (0.40)")

ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=18, ha="right", fontsize=8)
ax.set_ylabel("P(signal present)")
ax.set_title("ScamShield AI -- Strands Decider 2B risk signal probabilities")
ax.set_ylim(0, 1.05)
ax.legend(fontsize=8)

# colour verdict labels above each group
for i, v in enumerate(verdicts):
    label = v.replace("_", " ").upper()
    ax.text(x[i], max(unknown[i], urgency[i], unusual[i]) + 0.04, label,
            ha="center", va="bottom", fontsize=7, color=verdict_colors[v], fontweight="bold")

plt.tight_layout()
plt.savefig("scamshield_signals.png", dpi=150)
plt.show()
print("Chart saved to scamshield_signals.png")

## 5. The tools

Three Strands `@tool` functions the LLM agent can call. All data is synthetic — no external API needed.

In [ ]:
import re
from strands import tool

SCAM_PATTERNS = [
    {"id": "SP001", "name": "KYC Update Scam",      "keywords": ["kyc", "update", "bank account", "verify", "blocked"],           "risk": "high"},
    {"id": "SP002", "name": "Lottery / Prize Scam",  "keywords": ["won", "prize", "lottery", "claim", "reward"],                  "risk": "high"},
    {"id": "SP003", "name": "Impersonation Scam",    "keywords": ["police", "cbi", "income tax", "irs", "officer", "arrest"],      "risk": "high"},
    {"id": "SP004", "name": "Urgent Transfer Scam",  "keywords": ["urgent", "emergency", "immediately", "right now", "asap"],      "risk": "high"},
    {"id": "SP005", "name": "OTP / PIN Sharing",     "keywords": ["otp", "pin", "password", "share", "send me", "tell me"],       "risk": "high"},
    {"id": "SP006", "name": "Job Offer Scam",        "keywords": ["job", "work from home", "registration fee"],                   "risk": "high"},
    {"id": "SP007", "name": "Advance Fee Scam",      "keywords": ["advance", "processing fee", "deposit first", "pay first"],     "risk": "high"},
    {"id": "SP008", "name": "Suspicious UPI ID",     "keywords": ["unknown upi"],                                                 "risk": "medium"},
    {"id": "SP010", "name": "Normal Bill Payment",   "keywords": ["electricity", "water bill", "recharge", "dth"],               "risk": "low"},
]

SAFETY_TIPS = {
    "unknown_recipient": [
        "Always verify the recipient's identity through an official channel before sending money.",
        "Call the person directly on a known number - do not use the number they provided.",
    ],
    "urgency_pressure": [
        "Legitimate requests never require you to act within minutes - take your time.",
        "Urgency is a classic pressure tactic used in scams; slow down and verify.",
    ],
    "unusual_payment": [
        "Never share your OTP, PIN or password with anyone - not even bank staff.",
        "Gift cards and wire transfers are irreversible - be extra cautious.",
    ],
    "low_risk": [
        "Always double-check the recipient's UPI ID or account number before confirming.",
    ],
}


@tool
def analyze_transaction(description: str) -> dict:
    """Extract key risk indicators from a payment transaction description."""
    text = description.lower()
    amounts = re.findall(r'rs\.?\s*(\d[\d,]*)', text)
    large_amount = any(int(a.replace(',', '')) > 10000 for a in amounts if a.replace(',', '').isdigit())
    indicators = {
        "unknown_recipient": any(w in text for w in ["unknown", "stranger", "never met", "random"]),
        "urgency_mentioned": any(w in text for w in ["urgent", "immediately", "right now", "asap", "emergency"]),
        "large_amount":      large_amount,
        "otp_pin_requested": any(w in text for w in ["otp", "pin", "password", "passcode"]),
    }
    return {"indicators": indicators, "risk_signals_found": [k for k, v in indicators.items() if v]}


@tool
def lookup_scam_patterns(description: str) -> dict:
    """Look up known scam patterns that match keywords in the transaction description."""
    text = description.lower()
    matched = []
    for p in SCAM_PATTERNS:
        hits = [kw for kw in p["keywords"] if kw in text]
        if hits:
            matched.append({"name": p["name"], "risk": p["risk"], "matched_keywords": hits})
    matched.sort(key=lambda x: {"high": 0, "medium": 1, "low": 2}[x["risk"]])
    return {"patterns_matched": matched, "highest_risk": matched[0]["risk"] if matched else "unknown"}


@tool
def get_safety_tips(flags: list) -> dict:
    """Return safety tips relevant to the risk flags found in a transaction."""
    tips = []
    for flag in flags:
        tips.extend(SAFETY_TIPS.get(flag, []))
    if not tips:
        tips = SAFETY_TIPS["low_risk"]
    seen, unique = set(), []
    for t in tips:
        if t not in seen:
            seen.add(t)
            unique.append(t)
    return {"tips": unique[:4]}


TOOLS = [analyze_transaction, lookup_scam_patterns, get_safety_tips]

# Quick tool smoke-test
print("analyze_transaction:", analyze_transaction("An unknown person asked me to pay Rs.15,000 urgently."))
print("lookup_scam_patterns:", lookup_scam_patterns("Income tax officer said pay immediately or face arrest."))
print("get_safety_tips:", get_safety_tips(["unknown_recipient", "urgency_pressure"]))

## 6. The full agent

Strands Decider 2B runs **before** the LLM sees the message. The verdict is injected into the
context so the LLM reply is consistent with the decision.

In [ ]:
import time
from strands import Agent
from decider_kit.llm import agent_model, default_model_id, strip_thinking

SYSTEM_PROMPT = """You are ScamShield AI, a payment scam detection assistant.

When a user describes a payment request or transaction:
1. Use analyze_transaction to extract risk indicators.
2. Use lookup_scam_patterns to check for known scam types.
3. Use get_safety_tips with the relevant risk flags.
4. Give a clear verdict: LOW RISK [safe], HIGH RISK [warning], or NEEDS REVIEW [check]
5. Explain the red flags in 2-3 sentences.
6. Share the most relevant safety tips.

Be direct and concise. Never alarm unnecessarily for low-risk transactions.
For Needs Review, ask one specific follow-up question."""


class ScamShieldAgent:
    """ScamShield AI: Strands Decider 2B policy check + Strands LLM agent."""

    def __init__(self):
        self.decider  = Decider()
        self.policy   = ScamPolicyCheck(self.decider)
        self.model_id = default_model_id()
        self._agent   = Agent(
            model=agent_model(self.model_id),
            system_prompt=SYSTEM_PROMPT,
            tools=TOOLS,
            callback_handler=None,
        )

    def chat(self, message: str) -> dict:
        # Step 1: Decider classifies risk BEFORE the LLM
        decision = self.policy.check(message)

        hint_map = {
            "high_risk":    "[HIGH RISK] - scam indicators detected.",
            "needs_review": "[NEEDS REVIEW] - some risk signals present; ask for more detail.",
            "low_risk":     "[LOW RISK] - no major scam indicators found.",
        }
        augmented = (
            f"{message}\n\n"
            f"[ScamShield policy check: {hint_map[decision['verdict']]} "
            f"Signals: unknown_recipient={decision['unknown_recipient']:.2f}, "
            f"urgency={decision['urgency_pressure']:.2f}, "
            f"unusual={decision['unusual_payment']:.2f}]"
        )

        # Step 2: LLM composes the human-friendly reply
        t0 = time.perf_counter()
        result = self._agent(augmented)
        reply  = strip_thinking(str(result))
        latency_ms = round((time.perf_counter() - t0) * 1000)

        return {
            "verdict":    decision["verdict"],
            "flags":      decision["flags"],
            "confidence": decision["confidence"],
            "decider_ms": decision["decider_ms"],
            "reply":      reply,
            "latency_ms": latency_ms,
        }


agent = ScamShieldAgent()
print(f"Agent ready. LLM: {agent.model_id}")

## 7. Interactive demo

Try the five demo transactions and see the full reply with decision details.

In [ ]:
demo_transactions = [
    "I received a payment request from an unknown person asking me to transfer Rs.15,000 urgently.",
    "My friend Priya asked me to pay her Rs.500 for lunch via UPI, she is a saved contact.",
    "Someone claiming to be from the Income Tax department said I owe Rs.50,000 and must pay right now or face arrest.",
    "A work-from-home recruiter I found online is asking for a Rs.500 registration fee before my first task.",
    "Paying my electricity bill of Rs.1,340 through the official BESCOM portal.",
]

VERDICT_ICON = {"high_risk": "RISK", "needs_review": "REVIEW", "low_risk": "SAFE"}

for tx in demo_transactions[:2]:   # run 2 to avoid rate limits; remove [:2] for all 5
    print("=" * 70)
    print(f"USER: {tx}")
    out = agent.chat(tx)
    icon = VERDICT_ICON[out['verdict']]
    print(f"\n[{icon}] Decider: confidence={out['confidence']:.2f}, flags={out['flags']}, {out['decider_ms']} ms")
    print(f"\nSCAMSHIELD: {out['reply']}")
    print(f"\n(total latency: {out['latency_ms']} ms)")

## 8. Benchmark: decider vs LLM latency

Reproduce the key finding: Strands Decider 2B answers in ~3 s; the LLM takes ~20 s for the same questions.
Numbers below come from the real bench run (`./kit bench --split tune`).

In [ ]:
import matplotlib.pyplot as plt

# Real numbers from submission/bench/results.json (tune split, 17 cases)
bench = {
    "system":        ["Strands Decider 2B", "LLM (qwen3.5-4b)"],
    "latency_p50":   [2990,  20727],
    "latency_p95":   [3090,  36488],
    "cost_per_1k":   [0.035, 0.000],
    "confident_pct": [57,    100],
}

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Latency
ax = axes[0]
colors = ["#4c72b0", "#dd8452"]
bars = ax.bar(bench["system"], bench["latency_p50"], color=colors, alpha=0.85)
ax.errorbar(
    bench["system"],
    bench["latency_p50"],
    yerr=[[0, 0], [b - a for a, b in zip(bench["latency_p50"], bench["latency_p95"])]],
    fmt="none", color="black", capsize=5
)
for bar, val in zip(bars, bench["latency_p50"]):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 300, f"{val:,} ms",
            ha="center", va="bottom", fontsize=9)
ax.set_ylabel("Latency p50 (ms)")
ax.set_title("Decision latency: 6.9x faster")
ax.set_ylim(0, 28000)

# Cost
ax = axes[1]
bars2 = ax.bar(bench["system"], bench["cost_per_1k"], color=colors, alpha=0.85)
for bar, val in zip(bars2, bench["cost_per_1k"]):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 0.0005,
            f"${val:.4f}" if val > 0 else "$0.00",
            ha="center", va="bottom", fontsize=9)
ax.set_ylabel("USD per 1,000 decisions")
ax.set_title("Cost per 1,000 decisions")

plt.suptitle("ScamShield AI -- Strands Decider 2B vs LLM baseline", fontsize=11)
plt.tight_layout()
plt.savefig("scamshield_benchmark.png", dpi=150)
plt.show()
print("Benchmark chart saved to scamshield_benchmark.png")
print(f"Speedup: {bench['latency_p50'][1] / bench['latency_p50'][0]:.1f}x faster than the LLM")

## 9. How to reuse the pattern

Drop the three questions into any Strands agent that needs language-based risk classification:

```python
from decider_kit import Decider

d = Decider()   # reads .env
a = d.ask(user_message, QUESTIONS)

if a["unknown_recipient"].p_yes > 0.60:
    # block or warn
elif a["unknown_recipient"].p_yes > 0.40:
    # ask a follow-up question
else:
    # proceed normally
```

**To adapt for another domain:** reword the option descriptions in `QUESTIONS`.
The question text and option text carry most of the signal — the model reads them at inference time.

Examples:
- **Phishing email classifier** — reword `unknown_recipient` to `unknown_sender`, `urgency_pressure` stays the same, `unusual_payment` becomes `unusual_request`.
- **Fake job offer detector** — reword for job context: upfront fees, vague role description, too-good salary.
- **Loan application fraud** — identity unverifiable, income claim inconsistent, document looks forged.

---

**ScamShield AI** | Decision pattern: `#policy` | Tags: `#strands-decider-2b` `#policy`

Built with the **AWS User Group Madurai** Strands Decider hackathon kit: https://www.awsugmdu.in  
Strands Decider 2B: https://github.com/strands-labs/strands-decider (Apache-2.0)  
Licence: Apache-2.0